# LLM Evaluation: Structured Data Extraction

Imagine your support team receives hundreds of customer emails every day about damaged items, missing deliveries, billing issues, and cancellations. To reduce manual work, you use an LLM to read each message and extract key information in a structured JSON format, including the order ID, issue type, requested action, and customer email.

However, not every model output is production-ready. Some responses are not valid JSON, some are missing required fields, and others contain incorrect or inconsistent values.

To make this system reliable, you introduce a set of automated evaluation checks. The output must be valid JSON, follow the required schema, include all mandatory fields, and use correct value types.

In this example, we will compare reference-free schema validation with reference-based evaluation. The important lesson is not only whether an output failed, but where it failed: JSON parsing, schema validation, type checks, or value accuracy.


In [21]:
# Import libraries
import re
import json

import pandas as pd

from sklearn.metrics import classification_report

In [22]:
pd.set_option('display.max_colwidth', None)

## 1. Synthetic Dataset

We will create a static dataset of customer support emails and corresponding structured outputs. Each example contains the original message, the expected reference JSON, and the model's generated output.

In [23]:
# Static synthetic dataset

synthetic_data = [
    {
        "id": 1,
        "label": "correct",
        "input_text": "Order A1047 arrived damaged. The hinge is cracked and I would prefer a refund instead of a replacement. Please reply to john.smith@example.com.",
        "reference_json": {
            "order_id": "A1047",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "john.smith@example.com"
        },
        "model_output": {
            "order_id": "A1047",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "john.smith@example.com"
        }
    },
    {
        "id": 2,
        "label": "correct",
        "input_text": "My order B2099 shows delivered but nothing arrived. Please resend it. Contact me at anna.k@mailer.com.",
        "reference_json": {
            "order_id": "B2099",
            "issue_type": "not_delivered",
            "requested_action": "resend",
            "customer_email": "anna.k@mailer.com"
        },
        "model_output": {
            "order_id": "B2099",
            "issue_type": "not_delivered",
            "requested_action": "resend",
            "customer_email": "anna.k@mailer.com"
        }
    },
    {
        "id": 3,
        "label": "correct",
        "input_text": "I received the wrong size for order C3301. I want to exchange it. tom.w@domain.com.",
        "reference_json": {
            "order_id": "C3301",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "tom.w@domain.com"
        },
        "model_output": {
            "order_id": "C3301",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "tom.w@domain.com"
        }
    },
    {
        "id": 4,
        "label": "correct",
        "input_text": "Please cancel order D4410 before shipment. kate.jones@mail.com.",
        "reference_json": {
            "order_id": "D4410",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "kate.jones@mail.com"
        },
        "model_output": {
            "order_id": "D4410",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "kate.jones@mail.com"
        }
    },
    {
        "id": 5,
        "label": "correct",
        "input_text": "Order E5522 was charged twice. Refund the extra payment please. sam.billing@mail.com.",
        "reference_json": {
            "order_id": "E5522",
            "issue_type": "billing_issue",
            "requested_action": "refund",
            "customer_email": "sam.billing@mail.com"
        },
        "model_output": {
            "order_id": "E5522",
            "issue_type": "billing_issue",
            "requested_action": "refund",
            "customer_email": "sam.billing@mail.com"
        }
    },
    {
        "id": 6,
        "label": "correct",
        "input_text": "Order F6008 has not arrived yet. Could you investigate the delay? lisa.p@mail.com.",
        "reference_json": {
            "order_id": "F6008",
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "lisa.p@mail.com"
        },
        "model_output": {
            "order_id": "F6008",
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "lisa.p@mail.com"
        }
    },
    {
        "id": 7,
        "label": "correct",
        "input_text": "Order G7120 arrived cracked. I would like a refund. mike.r@mail.com.",
        "reference_json": {
            "order_id": "G7120",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "mike.r@mail.com"
        },
        "model_output": {
            "order_id": "G7120",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "mike.r@mail.com"
        }
    },
    {
        "id": 8,
        "label": "correct",
        "input_text": "Order H8804 wrong color received. Exchange please. amy.customer@mail.com.",
        "reference_json": {
            "order_id": "H8804",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "amy.customer@mail.com"
        },
        "model_output": {
            "order_id": "H8804",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "amy.customer@mail.com"
        }
    },
    {
        "id": 9,
        "label": "correct",
        "input_text": "Order I9013 arrived damaged and I prefer a refund. bob.h@mail.com.",
        "reference_json": {
            "order_id": "I9013",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "bob.h@mail.com"
        },
        "model_output": {
            "order_id": "I9013",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "bob.h@mail.com"
        }
    },
    {
        "id": 10,
        "label": "correct",
        "input_text": "Cancel order J1022 before it ships. alice.cancel@mail.com.",
        "reference_json": {
            "order_id": "J1022",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "alice.cancel@mail.com"
        },
        "model_output": {
            "order_id": "J1022",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "alice.cancel@mail.com"
        }
    },
    {
        "id": 11,
        "label": "correct",
        "input_text": "Order K1140 shows delivered but nothing arrived. Please investigate first. jack.inbox@mail.com.",
        "reference_json": {
            "order_id": "K1140",
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "jack.inbox@mail.com"
        },
        "model_output": {
            "order_id": "K1140",
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "jack.inbox@mail.com"
        }
    },
    {
        "id": 12,
        "label": "correct",
        "input_text": "Cancel order L1207. It was an accidental duplicate. emma.dup@mail.com.",
        "reference_json": {
            "order_id": "L1207",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "emma.dup@mail.com"
        },
        "model_output": {
            "order_id": "L1207",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "emma.dup@mail.com"
        }
    },
    {
        "id": 13,
        "label": "correct",
        "input_text": "Order M1333 charged twice. Refund the extra payment. paul.pay@mail.com.",
        "reference_json": {
            "order_id": "M1333",
            "issue_type": "billing_issue",
            "requested_action": "refund",
            "customer_email": "paul.pay@mail.com"
        },
        "model_output": {
            "order_id": "M1333",
            "issue_type": "billing_issue",
            "requested_action": "refund",
            "customer_email": "paul.pay@mail.com"
        }
    },
    {
        "id": 14,
        "label": "correct",
        "input_text": "Order N1402 wrong model received. Exchange please. zoe.swap@mail.com.",
        "reference_json": {
            "order_id": "N1402",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "zoe.swap@mail.com"
        },
        "model_output": {
            "order_id": "N1402",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "zoe.swap@mail.com"
        }
    },
    {
        "id": 15,
        "label": "wrong",
        "input_text": "Order O1555 arrived damaged. I want a refund. nina.damage@mail.com.",
        "reference_json": {
            "order_id": "O1555",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "nina.damage@mail.com"
        },
        "model_output": "{\"order_id\":\"O1555\",\"issue_type\":\"damaged_item\",\"requested_action\":\"refund\",\"customer_email\":\"nina.damage@mail.com\""
    },
    {
        "id": 16,
        "label": "wrong",
        "input_text": "Order P1609 has not arrived. Please investigate. sara.late@mail.com.",
        "reference_json": {
            "order_id": "P1609",
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "sara.late@mail.com"
        },
        "model_output": "{\"order_id\":\"P1609\", \"issue_type\":\"not_delivered\" \"requested_action\":\"investigate\", \"customer_email\":\"sara.late@mail.com\"}"
    },
    {
        "id": 17,
        "label": "wrong",
        "input_text": "Order Q1777 wrong finish received. Exchange please. li.finish@mail.com.",
        "reference_json": {
            "order_id": "Q1777",
            "issue_type": "wrong_item",
            "requested_action": "exchange",
            "customer_email": "li.finish@mail.com"
        },
        "model_output": "Here is the JSON you asked for: {\"order_id\":\"Q1777\",\"issue_type\":\"wrong_item\",\"requested_action\":\"exchange\",\"customer_email\":\"li.finish@mail.com\"}"
    },
    {
        "id": 18,
        "label": "wrong",
        "input_text": "Order R1881 arrived cracked. I would like a refund. rita.cracked@mail.com.",
        "reference_json": {
            "order_id": "R1881",
            "issue_type": "damaged_item",
            "requested_action": "refund",
            "customer_email": "rita.cracked@mail.com"
        },
        "model_output": "[{\"order_id\":\"R1881\",\"issue_type\":\"damaged_item\",\"requested_action\":\"refund\",\"customer_email\":\"rita.cracked@mail.com\"}]"
    },
    {
        "id": 19,
        "label": "wrong",
        "input_text": "Cancel order S1990 before it ships. tom.travel@mail.com.",
        "reference_json": {
            "order_id": "S1990",
            "issue_type": "cancellation_request",
            "requested_action": "cancel_order",
            "customer_email": "tom.travel@mail.com"
        },
        "model_output": {
            "order_id": "S1990",
            "issue_type": "cancellation_request",
            "customer_email": "tom.travel@mail.com"
        }
    },
    {
        "id": 20,
        "label": "wrong",
        "input_text": "Order T2006 partially delivered. Please investigate. ops.team@mail.com.",
        "reference_json": {
            "order_id": "T2006",
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "ops.team@mail.com"
        },
        "model_output": {
            "order_id": 2006,
            "issue_type": "not_delivered",
            "requested_action": "investigate",
            "customer_email": "ops.team@mail.com"
        }
    }
]


In [24]:
df = pd.DataFrame(
    synthetic_data,
    columns=["id", "label", "input_text", "reference_json", "model_output"]
)

label_summary = (
    df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_summary["share_%"] = (label_summary["count"] / label_summary["count"].sum() * 100).round(1)

label_summary

,label,count,share_%
0,correct,14,70.0
1,wrong,6,30.0


## 2. Reference-free checks

In [25]:
REQUIRED_KEYS = {"order_id", "issue_type", "requested_action", "customer_email"}

EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

ORDER_ID_RE = re.compile(r"^[A-Z]\d{4}$")  # e.g., A1047

In [26]:
#helper
def parse_json_object(x):
    try:
        obj = x if isinstance(x, dict) else json.loads(x)
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

#checks
def is_valid_json(model_output):
    return parse_json_object(model_output) is not None

def has_required_keys(model_output):
    obj = parse_json_object(model_output)
    return obj is not None and REQUIRED_KEYS.issubset(obj.keys())

def has_no_extra_keys(model_output):
    obj = parse_json_object(model_output)
    return obj is not None and set(obj.keys()).issubset(REQUIRED_KEYS)

def types_and_patterns_valid(model_output):
    obj = parse_json_object(model_output)
    if obj is None or not REQUIRED_KEYS.issubset(obj.keys()):
        return False

    # type checks
    if not all(isinstance(obj[k], str) for k in REQUIRED_KEYS):
        return False

    # order id pattern
    if not ORDER_ID_RE.match(obj["order_id"]):
        return False

    # email pattern (allow empty)
    email = obj["customer_email"]
    if email != "" and not EMAIL_RE.match(email):
        return False

    return True

In [27]:
df["valid_json"] = df["model_output"].apply(is_valid_json)
df["all_required_keys"] = df["model_output"].apply(has_required_keys)
df["no_extra_keys"] = df["model_output"].apply(has_no_extra_keys)
df["correct_types_and_patterns"] = df["model_output"].apply(types_and_patterns_valid)

df.tail()

,id,label,input_text,reference_json,model_output,valid_json,all_required_keys,no_extra_keys,correct_types_and_patterns
15,16,wrong,Order P1609 has not arrived. Please investigate. sara.late@mail.com.,"{'order_id': 'P1609', 'issue_type': 'not_delivered', 'requested_action': 'investigate', 'customer_email': 'sara.late@mail.com'}","{""order_id"":""P1609"", ""issue_type"":""not_delivered"" ""requested_action"":""investigate"", ""customer_email"":""sara.late@mail.com""}",False,False,False,False
16,17,wrong,Order Q1777 wrong finish received. Exchange please. li.finish@mail.com.,"{'order_id': 'Q1777', 'issue_type': 'wrong_item', 'requested_action': 'exchange', 'customer_email': 'li.finish@mail.com'}","Here is the JSON you asked for: {""order_id"":""Q1777"",""issue_type"":""wrong_item"",""requested_action"":""exchange"",""customer_email"":""li.finish@mail.com""}",False,False,False,False
17,18,wrong,Order R1881 arrived cracked. I would like a refund. rita.cracked@mail.com.,"{'order_id': 'R1881', 'issue_type': 'damaged_item', 'requested_action': 'refund', 'customer_email': 'rita.cracked@mail.com'}","[{""order_id"":""R1881"",""issue_type"":""damaged_item"",""requested_action"":""refund"",""customer_email"":""rita.cracked@mail.com""}]",False,False,False,False
18,19,wrong,Cancel order S1990 before it ships. tom.travel@mail.com.,"{'order_id': 'S1990', 'issue_type': 'cancellation_request', 'requested_action': 'cancel_order', 'customer_email': 'tom.travel@mail.com'}","{'order_id': 'S1990', 'issue_type': 'cancellation_request', 'customer_email': 'tom.travel@mail.com'}",True,False,True,False
19,20,wrong,Order T2006 partially delivered. Please investigate. ops.team@mail.com.,"{'order_id': 'T2006', 'issue_type': 'not_delivered', 'requested_action': 'investigate', 'customer_email': 'ops.team@mail.com'}","{'order_id': 2006, 'issue_type': 'not_delivered', 'requested_action': 'investigate', 'customer_email': 'ops.team@mail.com'}",True,True,True,False


In [28]:
def get_rf_predicted_error(row):
    if not row["valid_json"]:
        return "invalid_json"

    if not row["all_required_keys"]:
        return "missing_field"

    if not row["no_extra_keys"]:
        return "extra_field"

    if not row["correct_types_and_patterns"]:
        return "wrong_type"

    return "correct"


df["rf_predicted_error"] = df.apply(
    get_rf_predicted_error,
    axis=1
)

df["rf_predicted_label"] = df["rf_predicted_error"].apply(
    lambda error_type: "correct" if error_type == "correct" else "wrong"
)


In [29]:
total_errors = (df["label"] != "correct").sum()
errors_caught = (df["rf_predicted_label"] == "wrong").sum()

print(f"Reference-free errors caught: {errors_caught} out of {total_errors}")


Reference-free errors caught: 6 out of 6


In [30]:
accuracy = (
    df["label"] == df["rf_predicted_label"]
).mean()

print("Reference-free checks accuracy:", round(accuracy * 100, 1), "%")


Reference-free checks accuracy: 100.0 %


In [31]:
print("Reference-free checks precision/recall:\n")
print(
    classification_report(
        df["label"],
        df["rf_predicted_label"],
        digits=3
    )
)


Reference-free checks precision/recall:

              precision    recall  f1-score   support

     correct      1.000     1.000     1.000        14
       wrong      1.000     1.000     1.000         6

    accuracy                          1.000        20
   macro avg      1.000     1.000     1.000        20
weighted avg      1.000     1.000     1.000        20



## 3. Reference based checks

In [32]:
def exact_match_check(row):
    model_output = row["model_output"]
    reference = row["reference_json"]
    
    obj = parse_json_object(model_output)
    if obj is None:
        return "invalid_json"
    
    if obj != reference:
        return "wrong_value"
    
    return "correct"

In [33]:
df["is_exact_match"] = df.apply(
    exact_match_check,
    axis=1
)

In [34]:
def total_predicted_error(row):   
    #reference-free checks
    if row["rf_predicted_error"] != "correct":
        return row["rf_predicted_error"]
    
    #if reference-free checks passed, try reference-based 
    if row["is_exact_match"] != "correct":
        return row["is_exact_match"]
    
    return "correct"


In [35]:
df["total_predicted_error"] = df.apply(
    total_predicted_error,
    axis=1
)

df["total_predicted_label"] = df["total_predicted_error"].apply(
    lambda error_type: "correct" if error_type == "correct" else "wrong"
)


In [36]:
total_accuracy = (
    df["label"] ==
    df["total_predicted_label"]
).mean()

print("Total accuracy:", round(total_accuracy * 100, 1), "%")


Total accuracy: 100.0 %


In [37]:
print("Total checks precision/recall:\n")

print(
    classification_report(
        df["label"],
        df["total_predicted_label"],
        digits=3
    )
)


Total checks precision/recall:

              precision    recall  f1-score   support

     correct      1.000     1.000     1.000        14
       wrong      1.000     1.000     1.000         6

    accuracy                          1.000        20
   macro avg      1.000     1.000     1.000        20
weighted avg      1.000     1.000     1.000        20



## 4. Failure stage analysis

The target metric above tells us how many examples failed, but not where the failures happened. That is exactly why the basic reference-free checks are useful: they turn a binary `wrong` label into a debugging path.


In [38]:
df["structurally_valid"] = (
    df["valid_json"]
    & df["all_required_keys"]
    & df["no_extra_keys"]
    & df["correct_types_and_patterns"]
)

structural_validity = df["structurally_valid"].mean()
structurally_valid_df = df[df["structurally_valid"]]
value_accuracy_structurally_valid = (
    structurally_valid_df["is_exact_match"] == "correct"
).mean()

print(
    "Structural validity:",
    f"{structural_validity * 100:.1f}%",
    f"({df['structurally_valid'].sum()}/{len(df)})"
)
print(
    "Value accuracy among structurally valid outputs:",
    f"{value_accuracy_structurally_valid * 100:.1f}%",
    f"({(structurally_valid_df['is_exact_match'] == 'correct').sum()}/{len(structurally_valid_df)})"
)


Structural validity: 70.0% (14/20)
Value accuracy among structurally valid outputs: 100.0% (14/14)


In [39]:
def get_failure_stage(row):
    if not row["valid_json"]:
        return "1. JSON parsing"

    if not row["all_required_keys"]:
        return "2. Required keys"

    if not row["no_extra_keys"]:
        return "3. Extra keys"

    if not row["correct_types_and_patterns"]:
        return "4. Types and patterns"

    if row["is_exact_match"] != "correct":
        return "5. Value accuracy"

    return "passed"


df["failure_stage"] = df.apply(get_failure_stage, axis=1)

failure_stage_summary = (
    df[df["failure_stage"] != "passed"]
    .groupby("failure_stage")
    .size()
    .rename("count")
    .reset_index()
)

failure_stage_summary["share_of_failures_%"] = (
    failure_stage_summary["count"] / failure_stage_summary["count"].sum() * 100
).round(1)

failure_stage_summary


,failure_stage,count,share_of_failures_%
0,1. JSON parsing,4,66.7
1,2. Required keys,1,16.7
2,4. Types and patterns,1,16.7


In [40]:
df.loc[
    df["failure_stage"] != "passed",
    [
        "id",
        "label",
        "failure_stage",
        "rf_predicted_error",
        "model_output"
    ]
]


,id,label,failure_stage,rf_predicted_error,model_output
14,15,wrong,1. JSON parsing,invalid_json,"{""order_id"":""O1555"",""issue_type"":""damaged_item"",""requested_action"":""refund"",""customer_email"":""nina.damage@mail.com"""
15,16,wrong,1. JSON parsing,invalid_json,"{""order_id"":""P1609"", ""issue_type"":""not_delivered"" ""requested_action"":""investigate"", ""customer_email"":""sara.late@mail.com""}"
16,17,wrong,1. JSON parsing,invalid_json,"Here is the JSON you asked for: {""order_id"":""Q1777"",""issue_type"":""wrong_item"",""requested_action"":""exchange"",""customer_email"":""li.finish@mail.com""}"
17,18,wrong,1. JSON parsing,invalid_json,"[{""order_id"":""R1881"",""issue_type"":""damaged_item"",""requested_action"":""refund"",""customer_email"":""rita.cracked@mail.com""}]"
18,19,wrong,2. Required keys,missing_field,"{'order_id': 'S1990', 'issue_type': 'cancellation_request', 'customer_email': 'tom.travel@mail.com'}"
19,20,wrong,4. Types and patterns,wrong_type,"{'order_id': 2006, 'issue_type': 'not_delivered', 'requested_action': 'investigate', 'customer_email': 'ops.team@mail.com'}"


### Interpretation

If we looked only at total correctness, we would know that 30% of outputs failed. That is useful, but it does not tell us what to fix first.

The failure-stage breakdown shows a more actionable story: structural validity is only 70%, while value accuracy among structurally valid outputs is 100%. In other words, the model is extracting the right values whenever it produces usable JSON with the expected shape. The failures are happening before value comparison: mostly at JSON parsing, with a smaller number of schema/type problems.

This is a very realistic production failure mode. Even if correctness is the target metric, these basic reference-free checks help explain why correctness drops. If we evaluated blindly with only a correct/wrong label, we would know the system is failing, but not that the first fix should focus on output structure: JSON formatting, constrained decoding, stricter prompts, schema validation, or retry/repair logic before spending time on value-extraction quality.
